# Surface Observation Postprocessing with Machine Learning

**Authors**: [Jesper Dramsch](https://www.ecmwf.int/en/about/who-we-are/staff-profiles/jesper-dramsch), [Matthew Chantry](https://www.ecmwf.int/en/about/who-we-are/staff-profiles/matthew-chantry) and [Fenwick Cooper](https://www.physics.ox.ac.uk/our-people/cooperf) adapted by Bouke Hefting (WUR)

*This notebook was last tested and operational on 16/05/2026. Please [report any issues](https://github.com/ecmwf-training/2026-ml-esm-training/issues).*

<!-- :::{admonition} About
:class: note, dropdown -->
This notebook was originally created for ECMWF's [MOOC on Machine Learning in Weather and Climate](https://learning.ecmwf.int/course/view.php?id=47), and has been lightly updated and tested for the purposes of this [course](https://learning.ecmwf.int/course/view.php?id=99). The original notebook can be found [here](https://github.com/ecmwf-training/mooc-machine-learning-weather-climate/blob/main/tier_2/regression_decision_trees/random_forest.ipynb), and a worked example can be found [here](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m2/Surface_Observation_Prediction_in_Pytorch_solved.ipynb).
<!-- ::: -->

<!-- :::{admonition} Running this notebook
:class: tip, dropdown -->
This notebook can be run/accessed on the following free online platforms. Please note they are not officially supported by or linked with ECMWF.

[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecmwf-training/2026-ml-esm-training/blob/main/m2/Surface_Observation_Prediction_in_Pytorch.ipynb)
[![kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m2/Surface_Observation_Prediction_in_Pytorch.ipynb)
[![binder](https://mybinder.org/badge.svg)](https://mybinder.org/v2/gh/ecmwf-training/2026-ml-esm-training/main?labpath=m2/Surface_Observation_Prediction_in_Pytorch.ipynb)
[![github](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m2/Surface_Observation_Prediction_in_Pytorch.ipynb)
<!-- 
::: -->

## Introduction

In this session we will 
* Load and process real world temperature data
* Analyse this data by building a neural network model
* Fill in missing code to train and visualise the model structure
* Improve the model based on its performance

By following the steps in the code we will introduce main concepts and code for working with real world data and deep learning models

### Machine Learning for Forecasts?

We will use deep learning techniques to improve temperature forecasts by predicting the difference between the observed temperature at a weather station and the temperature forecasted by the ECMWF Integrated Forecast System [(IFS)](https://www.ecmwf.int/en/forecasts/documentation-and-support/changes-ecmwf-model). We already explored this task with a random forest model in Module 1 of the course, and now will study the advantages offered by using a neural network architecture instead.

### In this notebook
In this notebook, we are trying to predict the difference between the temperature observed at a weather station and the temperature forecasted by the IFS at the closest point on the grid to the station. To achieve this, we will look at several different predictors based on physical principles.

The steps we will follow in this notebook are:

- Prepare the environment and load the data
- Split temperature forecast error datasets in training and testing groups
- Standardise the data and apply a Pytorch dataloader
- Build two ML models based using the Adam optimiser
    - A Sequential model
    - A Functional model
- Training the model and quantifying the model's accuracy
- Visualize the model's performance with fictional prediction errors
- Repeat the above process with inclusion of the soil temperature variable

## Prepare your environment

The following packages are used in this notebook:

- `pytorch-lightning` for developing and deploying machine learning models
- `torch` for building and training neural networks
- `sklearn` for machine learning tools
- `numpy` for handling arrays and mathematical functions
- `matplotlib` for visualisation

In [ ]:
%pip install -q -r https://raw.githubusercontent.com/ecmwf-training/2026-ml-esm-training/main/requirements.txt

In [ ]:
# First import the tools we will use. Where possible it is always worth using
# existing tools, even if the mathematics are simple to write down yourself
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Include these if you do not want to see warnings rendered in the final notebook output (not recommended during coding)
import warnings
warnings.filterwarnings('ignore')

### Pytorch DataLoaders
We will use the package [Pytorch](https://pytorch.org/) to train our neural network. If you have never used Pytorch before, it is worth checking out some of its basic tutorials. For the purposes of this notebook: Pytorch uses so-called `Dataset` and `Dataloader` objects to give structure to the data it passes through its implemented models. There are many `Dataset` and `Dataloader` classes pre-defined in Pytorch, and we'll here use the `TensorDataset` to wrap our data, and pass it into a `DataLoader`. We import the necessary parts:

In [ ]:
import torch
import pytorch_lightning as pl

from torch import nn
from torch.nn import functional as F
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

Before going further, we also have to download a utility function for plotting. This is contained within a [plugin](https://github.com/mchantry/climetlab-mltc-surface-observation-postprocessing) to the [Climetlab](https://climetlab.readthedocs.io/en/latest/) package, which is no longer updated. In order to avoid a dependency on this package (which can cause errors in some environments), we load the function directly from the GitHub source code.

In [ ]:
# Download the raw Python file from GitHub
!wget -O utils.py https://raw.githubusercontent.com/mchantry/climetlab-mltc-surface-observation-postprocessing/master/climetlab_mltc_surface_observation_postprocessing/utils.py

# Run it in the notebook so functions are available
%run utils.py

## Loading the Data


We will now download the dataset: this comprises 36-hour forecast errors of 2m-temperature from a ECMWF's high resolution forecast system, using station observations as the truth. Currently there are three variables in the dataset:

- `forecast_error`: the difference between the forecasted value of 2m-temperature and the observed value, in °C. This is the target variable we wish to predict.
- `time_of_day`: the local time of day, in decimal hours. Useful for diagnosing the diurnal cycle model bias. This is a "feature", a variable we will use as input to the model we train to predict the forecast error.
- `soil_temperature`: the model soil temperature, in °C. This too is a feature.

For each variable, the dataset contains over 5 million datapoints covering around 8000 weather stations around the world (not specified in the dataset). We subsample this to 50,000 points for this exercise, to speed up training of our neural network. In a real setting, we would normally want to keep as much data as possible. However, also in such settings, it can be helpful to start with a small dataset so you can quickly iterate over training and check that your model is formulated properly (we get into debugging a model in more detail in the next submodule).

We will load each variable separately. Notice that we have to reshape the features to 2D arrays (of one column, with each row representing a data point) for compatability with `scikit-learn`. The target variable remains a 1D row vector.

In [ ]:
base_url = "https://object-store.os-api.cci1.ecmwf.int/sop/"

# load in variables one at a time (.reshape() puts in correct format for later)
forecast_error = np.genfromtxt(base_url + "forecast_error.csv", delimiter=",", skip_header=1).reshape(-1,1)
soil_temperature = np.genfromtxt(base_url + "soil_temperature.csv", delimiter=",", skip_header=1).reshape(-1, 1)
time_of_day = np.genfromtxt(base_url + "time_of_day.csv", delimiter=",", skip_header=1).reshape(-1, 1)

rng = np.random.default_rng(42)
idx = rng.choice(len(forecast_error), size=50_000, replace=False)
forecast_error = forecast_error[idx]
soil_temperature = soil_temperature[idx]
time_of_day = time_of_day[idx]

We have a hypothesis that the errors in the forecast can be easily explained by the time of day the measurements were taken. To test this, we will load the local time of day for when the measurements were recorded. Later we will add in the soil temperature, so we'll load this here to have the data loading step complete.

## Prepare the data

Now we can prepare the data for our model. This involves randomly splitting the data into training and testing sets, such that 80% of the data is in the training set and 20% in the testing set.

Ideally, we would like to know the geospatial location and time of our data points, because this would allow us to design our training/test sets to ensure independence (for more info, see Module 1.2 of the course). However, we don't have this information, so random sampling will be used.

The splitting into training and test sets can be done one in one call to the `train_test_split()` function of `scikit-learn`.

The function returns 6 outputs:

- `forecast_error_train`: it is the data containing errors in forecasted temperatures that will be used for training,
- `forecast_error_test`: it is the data containing errors in forecasted temperatures that will be used for testing,
- `time_of_day_train`: it is the data containing the local time of day when the measurements were taken that will be used for training,
- `time_of_day_test`: it is the data containing the local time of day when the measurements were taken that will be used for testing,
- `soil_temperature_train`: it is the data containing the soil temperature at the location where the measurements were taken that will be used for training,
- `soil_temperature_test`: it is the data containing the soil temperature at the location where the measurements were taken that will be used for testing.

To ensure our random split is the same each time the script is run, we set the `random_state` to 42.

In [ ]:
# Split each of the predictands and predictors in a random train/test split.
# We use 80% of the data for training & 20% for testing
(
    forecast_error_train,
    forecast_error_test,
    time_of_day_train,
    time_of_day_test,
    soil_temperature_train,
    soil_temperature_test,
) = train_test_split(
    forecast_error,
    time_of_day,
    soil_temperature,
    test_size=0.2,
    random_state=42,
)

## Data Preprocessing

The next step is to standardize the data. This is an important step that helps to prepare the data for training a neural network, especially in larger problems. To standardize the data, we will be using a method called `StandardScaler` method from scikit-learn library. This method divides each data point by the mean and standard deviation of the training dataset. By doing this, it ensures that the data has a mean of 0 and a standard deviation of 1.

The code below creates an instance of the `StandardScaler` class, and assigns it to the variable `scaler`. Then we use the `fit()` method of this `scaler` object, which calculates the mean and variance of the `time_of_day_train` data. Note that we fit this object on the training data only, since that is the mean and variance "known" to the training routine, and that the results are stored internally in the object. We next apply the learned transformation to the `time_of_day_train` data and create a new variable `time_of_day_train_norm` which contains the standardized data. The final line of code applies the same transformation to `time_of_day_test` data and creates a new variable `time_of_day_test_norm` which contains the standardized test data. Note that this only applies the standardization learned from the training data, and not from the test data.

It's worth noting that there are other ways to prepare data for a neural network, and sometimes a different approach may be more appropriate. For example, when working with precipitation data, it may be more appropriate to use a log transform before standardizing the data.

In [ ]:
scaler = StandardScaler()
scaler.fit(time_of_day_train)
print("Mean:", scaler.mean_, ", Variance:", scaler.var_)

time_of_day_train_norm = scaler.transform(time_of_day_train)
time_of_day_test_norm = scaler.transform(time_of_day_test)


Now, we must convert the `NumPy` arrays we just created to `PyTorch` tensors. This is done using the `torch.from_numpy()` function. The `.float()` at the end of the line is used to ensure the tensors are in a `float` data type. 

The next step is to create `TensorDataset`s from the data. The datasets are created by passing the `data` (here, input features to the model) and `labels` (here, target prediction error) tensors to the `TensorDataset` constructor. 

Finally, we create a `DataLoader` from the dataset. A `DataLoader` is a PyTorch utility that allows us to iterate over the dataset in batches. It takes the dataset and some options as input, such as the batch size and whether to shuffle the data. In this case, the batch size is set to 32, and the data is set to be shuffled before each epoch. We now will have a `DataLoader` that can be used to iterate over the data in batches during the training process. The `DataLoader` is useful for loading large datasets that do not fit into memory all at once, as it loads only a batch at a time.

The class below defines three dataloaders: For training, for validation (e.g. hyperparameter tuning) and for ultimate testing. Note that the validation data is created by splitting the training dataset again, along an 85%/15% split.

In [ ]:
class SOPDataModule(pl.LightningDataModule):
    def __init__(self, train_data, train_labels, test_data, test_labels, random_state=None):
        super().__init__()
        
        self.train_data, self.val_data, self.train_label, self.val_label = train_test_split(
           train_data, train_labels, test_size=0.15, random_state=random_state
        )
        self.test_data = test_data
        self.test_label = test_labels

        self.batch_size = 512
    
    def prepare_data(self):
        self.dataset_train = TensorDataset(torch.from_numpy(self.train_data).float(), torch.from_numpy(self.train_label).float())
        self.dataset_val = TensorDataset(torch.from_numpy(self.val_data).float(), torch.from_numpy(self.val_label).float())
        self.dataset_test = TensorDataset(torch.from_numpy(self.test_data).float(), torch.from_numpy(self.test_label).float())

    def train_dataloader(self):
        return DataLoader(self.dataset_train, batch_size=self.batch_size)

    def val_dataloader(self):
        return DataLoader(self.dataset_val, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.dataset_test, batch_size=self.batch_size)


In [ ]:
datamodule = SOPDataModule(time_of_day_train_norm, forecast_error_train, time_of_day_test_norm, forecast_error_test)

## Build Model

In Pytorch Lightning we commonly build models as classes. This code defines two classes: `FullyConnectedSequential` and `FullyConnectedFunctional`. Both functions create a model with a similar structure: it starts with an input layer, followed by several dense hidden layers, and ends with an output layer. 

### Sequential Model
We'll first briefly explain the sequential model's code (feel free to skip this explanation if you'd rather try to interpret the code itself). The `FullyConnectedSequential` model takes several parameters:

- `input_shape`: the number of input variables.
- `width`: how "wide" should each layer be, i.e. how many neurons it have.
- `depth`: how many layers the network should have.
- `activation`: the non-linear activation function to use (e.g. 'relu', 'sigmoid'...).
- `learning_rate`: the learning rate of the optimizer (default value is 10^-3).
- `final_activation`: the activation function for the final model layer (default value is 'linear').

These are then used to define several layers (combinations of `nn.Linear` and `activation`). Finally, in the line `self.model = ...`, we use the [`nn.Sequential`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Sequential.html) "container" to tell Pytorch to sequentially step through the list of provided input, hidden and output layers. Note that the output layer consists of application of `final_activation`, and then a linear transformation from `width` into a single (1) output: The forecasted prediction error.

The `training_step()` function is used to define the training loop. It takes a batch of data and the batch index, then it passes the data through the model, calculates the loss using the mean squared error loss function `F.mse_loss`, and logs the loss to the training log (the `validation_step()` and `test_step()` functions are similar. The `configure_optimizers` function is used to define the optimizer used to train the model. In this case, it is using the `Adam` optimizer with the learning rate that is passed to the class constructor.

In [ ]:
class FullyConnectedSequential(pl.LightningModule):
    def __init__(self, input_shape, width, depth, activation=None, learning_rate=10**(-3), final_activation=None):
        super().__init__()
        
        self.input_shape = input_shape
        self.width = width
        self.depth = depth

        self.learning_rate = learning_rate

        self.activation = activation() if activation is not None else nn.ReLU()
        self.final_activation = final_activation() if final_activation is not None else nn.Identity()

        self.hidden = []
        for _ in range(depth):
            self.hidden.append(self.activation)
            self.hidden.append(nn.Linear(width, width))

        self.model = nn.Sequential(nn.Linear(input_shape, width), *self.hidden, self.final_activation, nn.Linear(width, 1))

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y)
        self.log("validation_loss", loss)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y)
        self.log("test_loss", loss)
        return loss

    def configure_optimizers(self):
        opt = Adam(self.parameters(), lr=self.learning_rate)
        return opt

### Functional Model

This code is defining a PyTorch Lightning module called FullyConnectedFunctional. This module is similar to the FullyConnected module that you previously explained, but it uses a functional approach to define the model architecture: The input, hidden and output layers are defined as functions: each function takes in a tensor and applies the linear layer and activation functions to it and return the output tensor. The `forward` function is then used to define the forward pass of the model. Take a moment to compare this structure to how the previous model was defined.

In [ ]:
class FullyConnectedFunctional(pl.LightningModule):
    def __init__(self, input_shape, width, depth, activation=None, learning_rate=10**(-3), final_activation=None):
        super().__init__()
        
        self.input_shape = input_shape
        self.width = width
        self.depth = depth
        
        self.learning_rate = learning_rate

        self.activation = activation() if activation is not None else nn.ReLU()
        self.final_activation = final_activation() if final_activation is not None else nn.Identity()
        
        self.input = nn.Linear(input_shape, width)
        
        for i in range(self.depth):
            setattr(self, f"hidden{i}", nn.Linear(width, width))
        self.output = nn.Linear(width, 1)

    def forward(self, x):
        x = self.input(x)
        for i in range(self.depth):
            layer = getattr(self, f"hidden{i}")
            x = layer(self.activation(x))
        x = self.final_activation(x)
        return self.output(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y)
        self.log("validation_loss", loss)
        return loss
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y)
        self.log("test_loss", loss)
        return loss

    def configure_optimizers(self):
        opt = Adam(self.parameters(), lr=self.learning_rate)
        return opt

#### Adam Optimiser

When training a neural network, we use an optimization algorithm to adjust the parameters of the model in order to minimize the loss function. Notice that the optimiser we use is called `Adam` instead of the classic gradient descent algorithm described during the lessons.

Gradient descent is a simple optimization algorithm that updates the model's parameters by taking a step in the direction of the negative gradient of the loss function. The step size is determined by a learning rate parameter, which controls how large the step should be. Gradient descent can work well for simple models, but can be slow to converge for more complex models.

[Adam](https://arxiv.org/abs/1412.6980), on the other hand, is a more advanced optimization algorithm that is based on gradient descent. It uses an adaptive learning rate, which means that the step size is adjusted for each parameter based on the historical gradient information. This can help the optimization converge faster and more efficiently. Adam also includes additional features (called "momentum" and "RMSprop"), which can further improve the optimization process. Choosing Adam can improve the optimization process and make it converge faster and more efficiently.

### Creating the Model
Now we can create neural network object. You can use either the functional or sequential function defined above, since they end up doing the same. You can also try changing the depth and see what the summary says.

In [ ]:
model = FullyConnectedSequential(
    time_of_day_train_norm.shape[1], depth=3, width=32, activation=nn.Tanh, learning_rate=10 ** (-3)
)

pl.utilities.model_summary.ModelSummary(model, max_depth=-1)

Note: We have 3,300 parameters to learn. This is more than an order of magnitude fewer than the number of examples we retained (34,000 are now used in training), so we should not be too worried about overfitting.

## Model training

The `fit` function implements the training loop for a model in Pytorch lightning. 

The first argument passed to the `fit` function is the model. In this case, the training data is `time_of_day_train_norm` and the labels are `forecast_error_train`. These are the inputs and outputs that the model will use to learn during training as supplied by the dataloader.

The `epochs` argument is the number of times the model will cycle through the data.

Once the training is done, the model will be optimized to make predictions on new data with the same features.

In [ ]:
try:
    trainer = pl.Trainer(max_epochs=3, accelerator='gpu', devices=1)
except:
    print("No GPU found, training on CPU.")
    trainer = pl.Trainer(max_epochs=3)

trainer.fit(model, datamodule)

It is worth noting that training normally takes much longer than for this example, if the datasets and models are larger.

## Model Evaluation

Finally we can do a numerical evaluation how well the model performs on unseen data. As a teaching device, we'll also evaluate the model on training data. This number is relatively meaningless standing by itself, as the model will always perform on training data as long as the model converged. The validation data is unseen as training, but was involved in the optimization process, which can have indirect optimization consequences, by adjusting the architecture manually and generally making design choices based on the validation results. But it's good to learn why we test on actual unseen data and use the training value as the lower bound for our expectations of the final evaluation of the test data.

To evaluate the model performance, we will check its loss. Remember that we set this to be the mean squared error when we defined the model a few cells back. So as a baseline, we first calculate the mean squared error on the *uncorrected forecast dataset*:

In [ ]:
# Create a zero array with the same shape as the forecast error test dataset, this is the reference_array.
zero_test = 0.0 * forecast_error_test

# Calculate the mean squared error of the forecast error
baseline_mse =  metrics.mean_squared_error(zero_test, forecast_error_test)

# Print these statistics
print("MSE of Uncorrected forecast:", baseline_mse)

### Task: Compute the model loss
Now fill in the `???` spaces in the code cell below, to compute the mean squared error of the model, when evaluated on:
- The training dataset
- The validation dataset
- The hitherto unseen test dataset

To make this evaluation, we use Pytorch's implemented `trainer.test(dataloaders=datamodule.???_dataloader())`, where you must pick the correct `dataloader` from the `datamodule`.

In [ ]:
# Compute the model loss of the training dataset
train_loss = trainer.test(dataloaders=datamodule.???_dataloader())

# Compute the model loss of the model validation dataset
val_loss = ???

# Compute the model loss of the model test dataset. This is to test the model on true unseen data
test_loss = ???

print("Validation loss:", val_loss)
print("Train loss:", train_loss)
print("Test loss:", test_loss)

As you can see, training a neural network to predict the forecasting errors does a better job than not doing anything (the `test_loss` is lower than the `baseline_loss`), but the margin is not large. In fact, it is probably within the noise range of the dataset, since the `val_loss` is actually higher than the `baseline_loss`. To improve on these predictions, we therefore evaluate the model a little further.

### Visualization for Evaluation

We will now create some fake data to cover the entire day and get predictions to visualize the entire space of the data.  This is not always possible, but that's what we have this tutorial data for.

Below we create an array of time of day values spanning from 0 to 24, with a step of 0.001. This is a full range of time of day values that we want to make predictions on. The array is then reshaped to have a single axis, this is done with the `[..., np.newaxis]` slice. Then we apply the same standardization procedure (using the fitted `scaler` object) on the full range of time of day values as the one used on the training and test data, to ensure that the new data has the same scale as the data the model was trained on. Finally, the code uses the in-built model's function to make predictions on the normalized full range of time of day values. The predict function returns an array of predicted forecast errors, one for each input time of day value. The variable `forecast_error_full_range_predicted` stores these prediction.

In [ ]:
time_of_day_full_range = np.arange(0, 24, 0.001)[..., np.newaxis]
time_of_day_full_range_norm = scaler.transform(time_of_day_full_range)
forecast_error_full_range_predicted = model(torch.Tensor(time_of_day_full_range_norm)).detach().numpy()

In [ ]:
#Create an image representing time of day and the forecast error
tod_buffer, ax_extent, count = imgBufferFromVectors(
    time_of_day_test, forecast_error_test, nx=256, ny=256, extent=[], calc_average=False
)

In [ ]:
plt.imshow(np.log((count == 0.0) + count), cmap="Blues", origin="lower", extent=ax_extent, aspect="auto")

plt.xlim([0, 24])
plt.grid()
plt.xlabel("Time of day")
plt.ylabel("2m temperature error ($^\mathrm{o}$C)")
cb = plt.colorbar()
cb.set_label("Log( number of measurements )")

# Line of best fit
plt.plot(time_of_day_full_range, forecast_error_full_range_predicted, "red")

plt.show()

Although the model (red line) is better than nothing (and does indeed have the potential to improve our forecasts slightly), there is still a lot of of variation which is *not* explained. Indeed, the error only seems to have a weak relationship with the time of day.

### Task: Think about it
Does this look like a sensible model? Before moving on, think about the following: The plot shows a rather complex relationship between time of day and error. Should we expect our neural network to be able to capture this complexity? How can we improve the results? Let's test whether another predictor can help. We will try using the soil temperature.

## Adding more predictors

We will add a second predictor, the model soil temperature. Let's look at how the forecast error varies with these predictors to understand how well we might expect our model to perform.


In [ ]:
# Make image of the error with the new predictor
buffer, ax_extent, count = imgBufferFromVectors(
    soil_temperature_test, time_of_day_test, forecast_error_test, 128, 256, extent=[], calc_average=True
)

# Plot the image of the error
plt.imshow(buffer, vmin=-5, vmax=5, cmap="seismic", origin="lower", extent=ax_extent, aspect="auto")

plt.grid()
plt.xlabel("Soil temperature ($^o$C)")
plt.ylabel("Local time of day (hours)")
cb = plt.colorbar()
cb.set_label("Forecast - Observation ($^o$C)")

plt.show()


This plot is a heatmap of the forecast error (represented by the colour bar), plotted against the soil temperature and time of day (x and y axes, respectively). We see that forecast errors do indeed vary (noisily) with both of these predictors, so we can expect our model to improve if we add soil temperature as a predictor. It also appears that there is a more discernable pattern in the forecast error, and this implies that it might more be more feasible to model it.

### Task: Train the model with two predictors
In the code cell below, fill in the `???` blanks to re-train the model with these two predictors. You can use the cell where we first trained the model as inspiration.

The data for the two predictors is first concatenated and then standardized using `StandardScaler` again. Then we construct a functional model with depth and width of 3 and 32 respectively, using the `tanh` activation function.

In [ ]:
# Create the input features
X_train = np.concatenate([time_of_day_train, soil_temperature_train], axis=-1)
X_test = np.concatenate([time_of_day_test, soil_temperature_test], axis=-1)

# Code the solution to standardise the data
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# Construct the Data Module
datamodule = SOPDataModule(X_train, forecast_error_train, X_test, forecast_error_test)

# Construct a model, this time with 2 predictors
model_with_soil = FullyConnectedFunctional(
    X_train.shape[1], depth=3, width=32, activation=nn.Tanh, learning_rate=10 ** (-3)
)

#Train the model in the code below and fit the model
try:
    trainer = ???
except:
    print("No GPU found, training on CPU.")
    trainer = ???

trainer.fit(model_with_soil, datamodule)

### Evaluating the New Model
Let's again evaluate the model on the test data. In the code cell below, we calculate the mean absolute error of the new corrected forecast.

In [ ]:
# Use the error statistics computed above from uncorrected forecast dataset
forecast_corrected = forecast_error_test - model_with_soil(torch.Tensor(X_test)).detach().numpy()
print("Mean Squared Error Corrected:", metrics.mean_squared_error(zero_test, forecast_corrected))

By using two predictors, we have impoved the mean squared error in the 2m-temperature forecast by ~0.1K$^2$. Not nothing, but also not a big improvement.

#### Visual Evaluation

Let's now see how the model corrects the forecast across the full 2D space. We'll run the trained model over a range of values for the input variables `soil_temperature` and `time_of_day`. It first defines the range of values for these variables using `nx` and `ny`, which are the dimensions of the input buffer, and the `ax_extent` which is the range of values observed in the data. Then it creates the input data by stacking the meshgrid of these variable ranges using `np.meshgrid`, and transforms this input data using the `scaler` that was fit earlier on the training data. Then we predict the output using this input data and reshapes this output back to 2D plot using `raw_pred.reshape(input_buffer.shape[:-1])` for us to plot!

In [ ]:
# Run the fit model over the plot domain

# The x and y values of each point in the plot image, this covers
# the range of data observed in the data
nx = buffer.shape[0]
ny = buffer.shape[1]
x_st = np.linspace(ax_extent[0], ax_extent[1], nx)  # Represents soil_temperature
y_tod = np.linspace(ax_extent[2], ax_extent[3], ny)  # Represents time_of_day

# Create the input data
input_buffer = np.stack(np.meshgrid(y_tod, x_st), axis=-1)
X_plot = scaler.transform(input_buffer.reshape(-1, X_train.shape[-1]))

# Predict and reshape prediction back to 2D plot
raw_pred = model_with_soil(torch.Tensor(X_plot)).detach().numpy()
model_buffer = raw_pred.reshape(input_buffer.shape[:-1])


Let's compare the images. Notice that we're fixing the range of the plot between -5 and 5 to make these plots intuitively comparable.

In [ ]:
# Plot the image of the error
plt.imshow(buffer, vmin=-5, vmax=5, cmap="seismic", origin="lower", extent=ax_extent, aspect="auto")

plt.grid()
plt.title("Forecast Error or Raw Data")
plt.xlabel("Soil temperature ($^o$C)")
plt.ylabel("Local time of day (hours)")
cb = plt.colorbar()
cb.set_label("Forecast - Observation ($^o$C)")

plt.show()

# Plot the model output where we have data
plt.imshow(
    (model_buffer) * (count > 0), vmin=-5, vmax=5, cmap="seismic", origin="lower", extent=ax_extent, aspect="auto"
)

plt.grid()
plt.title("Model Output of Predicted Forecast Error")
plt.xlabel("Soil temperature ($^o$C)")
plt.ylabel("Local time of day (hours)")
cb = plt.colorbar()
cb.set_label("Neural Network model ($^o$C)")

plt.show()

# Plot the model over the whole domain
plt.imshow(model_buffer, vmin=-5, vmax=5, cmap="seismic", origin="lower", extent=ax_extent, aspect="auto")

plt.grid()
plt.title("Model Output of Extrapolating to the Full Domain")
plt.xlabel("Soil temperature ($^o$C)")
plt.ylabel("Local time of day (hours)")
cb = plt.colorbar()
cb.set_label("Neural Network model ($^o$C)")

plt.show()


## Conclusion

We have a prediction model that still contains errors that are barely better than not doing any prediction at all. We also see different model corrections for hour 23 and hour 0, when these should be closely correlated, as they’re mere seconds away from each other. And away from where data has been provided the model does not have constraints. We should not trust this part of the space without additional validation.

However, it does seem that the model has learned some basic features in the data, namely that the forecast error is on average negative in the afternoon and evening, and positive in the morning. Our model, consisting of a small number of parameters, is also still smooth, while the training data is very noisy. This also indicates that the model has not overfit the data. It has not learned the noise.

If we added predictors and size to our model, or kept on training for more epochs, perhaps we could fit the data better. But, our measurements of the forecast error are not perfect. There is some noise. We don’t want our model to capture this, and if we kept on adding complexity and fitting, the model will fit the noise, i.e. it will overfit the data. 

### Task (optional): How would you improve this model?
- Can you beat these predictions by changing the model architecture? Play around with activations, depth, width, learning rate of the neural network.
- Is there a way of building in any prior knowledge to even this simple setup? Perhaps you can encode the fact that 0 hour follows 24? Does this help the prediction?
- Would it have helped us to retain the full 5 million-sample dataset? Our raw data still has quite some "holes" and "noise" that might smooth away in a large-sample average.

Note that if you keep iterating, you are essentially doing a hyperparameter turning, for which we should ideally not use the test data as an evaluator. That data should be saved for a final evaluation once we are done designing the model.